In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import os
import time




In [2]:
df = pd.read_pickle('research\models\impute_result\df_complete_imputed.pkl')

print(f"Shape: {df.shape}")
print(f"\nColumns:\n{list(df.columns)}")
print(f"\nAny NaN: {df.isna().sum().sum()}")
print(f"\nTimestamp range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"\nDevice distribution:")
for d in ['device_pc1', 'device_pc2', 'device_pc3', 'device_pc4']:
    if d in df.columns:
        print(f"  {d}: {df[d].sum():.0f} rows")

Shape: (204942, 75)

Columns:
['timestamp', 'ping_ms', 'datarate', 'jitter', 'Latitude', 'Longitude', 'Altitude', 'speed_kmh', 'COG', 'precipIntensity', 'precipProbability', 'temperature', 'humidity', 'windSpeed', 'Traffic Jam Factor', 'Traffic Distance', 'Pos in Ref Round', 'measurement', 'area', 'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max', 'PCell_SNR_1', 'PCell_SNR_2', 'PCell_Downlink_Num_RBs', 'PCell_Downlink_TB_Size', 'PCell_Downlink_Average_MCS', 'PCell_Uplink_Num_RBs', 'PCell_Uplink_TB_Size', 'PCell_Uplink_Tx_Power_(dBm)', 'PCell_Downlink_frequency', 'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz', 'PCell_Band_Indicator', 'PCell_freq_MHz', 'scenario', 'target_datarate', 'operator', 'PCell_DL_RBs_MCS_Low', 'PCell_DL_RBs_MCS_Mid', 'PCell_DL_RBs_MCS_High', 'ping_ms_missing', 'datarate_missing', 'jitter_missing', 'Pos in Ref Round_missing', 'PCell_Cell_Identity_missing', 'PCell_Band_Indicator_missing', 'PCell_Uplink_frequency_missing', 'PCell_Downlink_bandwidth_

In [3]:
# Run this to see which columns still have NaN
nan_counts = df.isna().sum()
nan_cols = nan_counts[nan_counts > 0]
print(f"Columns with NaN ({len(nan_cols)}):")
for col, count in nan_cols.sort_values(ascending=False).items():
    print(f"  {col:<45} {count:>8,} ({count/len(df)*100:.2f}%)")

Columns with NaN (2):
  PCell_Downlink_frequency                        12,607 (6.15%)
  PCell_freq_MHz                                  10,031 (4.89%)


In [4]:
for col in df.select_dtypes(include=['float64', 'float32']).columns:
    unique_count = df[col].nunique()
    if unique_count > 20:
        # Check if the "real" values (from rows with no missing) are integers
        # Use rows where all _missing indicators are 0
        sample = df[col].dropna().head(1000)
        is_integer = (sample == sample.round()).all()
        if not is_integer:
            # Check if most values are round but some are not
            pct_round = (sample == sample.round()).mean()
            if pct_round > 0.5 and pct_round < 1.0:
                print(f"{col}:")
                print(f"  Unique: {unique_count}")
                print(f"  % integer-like: {pct_round*100:.1f}%")
                print(f"  Sample non-integer values: {sorted(sample[sample != sample.round()].unique()[:5])}")
                print()

PCell_Downlink_Average_MCS:
  Unique: 10507
  % integer-like: 98.6%
  Sample non-integer values: [np.float64(9.4815673828125), np.float64(9.654659271240234), np.float64(9.881030082702637), np.float64(10.408317565917969), np.float64(10.444368362426758)]

PCell_Downlink_bandwidth_MHz:
  Unique: 12385
  % integer-like: 97.1%
  Sample non-integer values: [np.float64(16.497224807739258), np.float64(16.738506317138672), np.float64(16.751323699951172), np.float64(16.75681495666504), np.float64(16.858938217163086)]

PCell_Uplink_bandwidth_MHz:
  Unique: 12372
  % integer-like: 97.1%
  Sample non-integer values: [np.float64(16.497224807739258), np.float64(16.738506317138672), np.float64(16.751323699951172), np.float64(16.75681495666504), np.float64(16.858938217163086)]

PCell_Band_Indicator:
  Unique: 12590
  % integer-like: 97.1%
  Sample non-integer values: [np.float64(5.124086380004883), np.float64(5.176089763641357), np.float64(5.192701816558838), np.float64(5.2882256507873535), np.float64(

In [5]:


# 1. Find valid values from the integer-like rows
for col in ['PCell_Downlink_Average_MCS', 'PCell_Downlink_bandwidth_MHz',
            'PCell_Uplink_bandwidth_MHz', 'PCell_Band_Indicator']:
    
    # Get the real valid values (the integer ones)
    all_vals = df[col].dropna()
    integer_vals = all_vals[all_vals == all_vals.round()]
    valid_set = sorted(integer_vals.unique())
    
    print(f"\n{col}:")
    print(f"  Valid values: {valid_set}")
    
    # Snap non-integer values to nearest valid value
    non_integer_mask = df[col] != df[col].round()
    n_fix = non_integer_mask.sum()
    
    if n_fix > 0:
        def snap_to_nearest(val):
            if val == round(val):
                return val
            return min(valid_set, key=lambda x: abs(x - val))
        
        df.loc[non_integer_mask, col] = df.loc[non_integer_mask, col].apply(snap_to_nearest)
        print(f"  Fixed {n_fix:,} values")
    
    # Verify
    remaining = (df[col] != df[col].round()).sum()
    print(f"  Remaining non-integer: {remaining}")


PCell_Downlink_Average_MCS:
  Valid values: [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(11.0), np.float64(12.0), np.float64(13.0), np.float64(14.0), np.float64(15.0), np.float64(16.0), np.float64(17.0), np.float64(18.0), np.float64(19.0), np.float64(20.0), np.float64(21.0), np.float64(22.0), np.float64(23.0), np.float64(24.0), np.float64(25.0), np.float64(26.0), np.float64(27.0), np.float64(28.0), np.float64(29.0)]
  Fixed 10,506 values
  Remaining non-integer: 0

PCell_Downlink_bandwidth_MHz:
  Valid values: [np.float64(5.0), np.float64(10.0), np.float64(15.0), np.float64(20.0)]
  Fixed 12,607 values
  Remaining non-integer: 0

PCell_Uplink_bandwidth_MHz:
  Valid values: [np.float64(5.0), np.float64(10.0), np.float64(15.0), np.float64(20.0)]
  Fixed 12,607 values
  Remaining non-integer: 0

PCell_Band_Indicator:
  Valid values: [n

In [6]:
# Forward fill categorical columns per device
categorical_to_fill = ['PCell_Downlink_frequency', 'PCell_freq_MHz', 'operator']

# Reconstruct device
device_series = pd.Series('unknown', index=df.index)
for d in ['device_pc1', 'device_pc2', 'device_pc3', 'device_pc4']:
    device_series[df[d] == 1] = d

for col in categorical_to_fill:
    n_before = df[col].isna().sum()
    df[col] = df.groupby(device_series)[col].transform(
        lambda x: x.ffill().bfill()
    )
    n_after = df[col].isna().sum()
    print(f"{col}: {n_before:,} → {n_after:,} NaN")



PCell_Downlink_frequency: 12,607 → 0 NaN
PCell_freq_MHz: 10,031 → 0 NaN
operator: 0 → 0 NaN


In [7]:
# Also drop _missing indicator columns (not needed for generation)
missing_cols = [c for c in df.columns if c.endswith('_missing')]
df = df.drop(columns=missing_cols)

# Final check
print(f"\nShape: {df.shape}")
print(f"Total NaN: {df.isna().sum().sum()}")
print(f"\nColumns ({len(df.columns)}):")
print(list(df.columns))

# Save
df.to_pickle('research\models\impute_result/df_complete_clean.pkl')
print(f"\nSaved: df_complete_clean.pkl")


Shape: (204942, 50)
Total NaN: 0

Columns (50):
['timestamp', 'ping_ms', 'datarate', 'jitter', 'Latitude', 'Longitude', 'Altitude', 'speed_kmh', 'COG', 'precipIntensity', 'precipProbability', 'temperature', 'humidity', 'windSpeed', 'Traffic Jam Factor', 'Traffic Distance', 'Pos in Ref Round', 'measurement', 'area', 'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max', 'PCell_SNR_1', 'PCell_SNR_2', 'PCell_Downlink_Num_RBs', 'PCell_Downlink_TB_Size', 'PCell_Downlink_Average_MCS', 'PCell_Uplink_Num_RBs', 'PCell_Uplink_TB_Size', 'PCell_Uplink_Tx_Power_(dBm)', 'PCell_Downlink_frequency', 'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz', 'PCell_Band_Indicator', 'PCell_freq_MHz', 'scenario', 'target_datarate', 'operator', 'PCell_DL_RBs_MCS_Low', 'PCell_DL_RBs_MCS_Mid', 'PCell_DL_RBs_MCS_High', 'device_pc1', 'device_pc2', 'device_pc3', 'device_pc4', 'direction_uplink', 'measured_qos_delay', 'hour', 'day_of_week', 'date']

Saved: df_complete_clean.pkl


In [10]:
pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.8/101.7 MB 2.5 MB/s eta 0:00:41
    --------------------------------------- 1.3/101.7 MB 2.7 MB/s eta 0:00:37
    --------------------------------------- 2.1/101.7 MB 2.8 MB/s eta 0:00:36
   - -------------------------------------- 2.6/101.7 MB 2.8 MB/s eta 0:00:36
   - -------------------------------------- 3.1/101.7 MB 2.8 MB/s eta 0:00:35
   - -------------------------------------- 3.9/101.7 MB 2.9 MB/s eta 0:00:34
   - -------------------------------------- 4.5/101.7 MB 2.9 MB/s eta 0:00:34
   -- ------------------------------------- 5.2/101.7 MB 2.9 MB/s eta 0:00:33
   -- ------------------------------------- 5.8/101.7 MB 3.0 MB/s eta 0:00:33
   -- ------------------------------------- 6.6/101.7 MB 3.0 MB/s eta 0:00:33
   -- ------------------------------------- 7.3/101.7 MB 3.0 MB/s eta 0:00:32


In [11]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error
import pickle
import warnings
import time
warnings.filterwarnings('ignore')

In [12]:
df = pd.read_pickle('research\models\impute_result\df_complete_clean.pkl')

In [13]:
df = df.sort_values('timestamp').reset_index(drop=True)

n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

df_train = df.iloc[:train_end].copy()
df_val = df.iloc[train_end:val_end].copy()
df_test = df.iloc[val_end:].copy()

print(f"Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")
print(f"Train period: {df_train['timestamp'].min()} to {df_train['timestamp'].max()}")
print(f"Val period:   {df_val['timestamp'].min()} to {df_val['timestamp'].max()}")
print(f"Test period:  {df_test['timestamp'].min()} to {df_test['timestamp'].max()}")

Train: 143459 | Val: 30741 | Test: 30742
Train period: 2021-06-22 09:49:54+02:00 to 2021-06-23 15:51:37+02:00
Val period:   2021-06-23 15:51:37+02:00 to 2021-06-24 10:19:02+02:00
Test period:  2021-06-24 10:19:02+02:00 to 2021-06-24 18:59:59+02:00


In [19]:


# Base input features (always available at generation time)
BASE_INPUTS = [
    'hour', 'day_of_week',
    'device_pc1', 'device_pc2', 'device_pc3', 'device_pc4',
    'direction_uplink',
    'measured_qos_delay',
    'measurement', 'operator'
]

# COG will be predicted as sin/cos then recovered
# We create sin_COG, cos_COG for training
for split_df in [df_train, df_val, df_test]:
    split_df['sin_COG'] = np.sin(np.radians(split_df['COG']))
    split_df['cos_COG'] = np.cos(np.radians(split_df['COG']))

In [20]:
# Causal chain levels — each level's outputs become inputs to subsequent levels
CAUSAL_CHAIN = [
    {
        'name': 'Level_0a_GPS',
        'targets': ['Latitude', 'Longitude'],
        'extra_inputs': [],  # only BASE_INPUTS
    },
    {
        'name': 'Level_0b_Mobility',
        'targets': ['speed_kmh', 'sin_COG', 'cos_COG', 'Altitude'],
        'extra_inputs': ['Latitude', 'Longitude'],
    },
    {
        'name': 'Level_0c_Weather',
        'targets': ['precipIntensity', 'precipProbability', 'temperature', 'humidity', 'windSpeed'],
        'extra_inputs': [],  # weather is mostly time-dependent
    },
    {
        'name': 'Level_0d_Traffic',
        'targets': ['Traffic Jam Factor', 'Traffic Distance'],
        'extra_inputs': ['Latitude', 'Longitude'],
    },
    {
        'name': 'Level_1_Signal',
        'targets': ['PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max',
                     'PCell_SNR_1', 'PCell_SNR_2', 'PCell_freq_MHz'],
        'extra_inputs': ['Latitude', 'Longitude', 'speed_kmh', 'sin_COG', 'cos_COG',
                         'Altitude', 'precipIntensity', 'precipProbability', 'temperature',
                         'humidity', 'windSpeed', 'Traffic Jam Factor', 'Traffic Distance'],
    },
    {
        'name': 'Level_2_CellConfig',
        'targets': ['PCell_Downlink_frequency', 'PCell_Band_Indicator',
                     'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz'],
        'extra_inputs': ['Latitude', 'Longitude', 'speed_kmh', 'sin_COG', 'cos_COG',
                         'Altitude', 'precipIntensity', 'precipProbability', 'temperature',
                         'humidity', 'windSpeed', 'Traffic Jam Factor', 'Traffic Distance',
                         'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max',
                         'PCell_SNR_1', 'PCell_SNR_2', 'PCell_freq_MHz'],
    },
    {
        'name': 'Level_3a_Downlink',
        'targets': ['PCell_Downlink_Average_MCS', 'PCell_Downlink_Num_RBs',
                     'PCell_Downlink_TB_Size',
                     'PCell_DL_RBs_MCS_Low', 'PCell_DL_RBs_MCS_Mid', 'PCell_DL_RBs_MCS_High'],
        'extra_inputs': ['Latitude', 'Longitude', 'speed_kmh', 'sin_COG', 'cos_COG',
                         'Altitude', 'Traffic Jam Factor', 'Traffic Distance',
                         'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max',
                         'PCell_SNR_1', 'PCell_SNR_2', 'PCell_freq_MHz',
                         'PCell_Downlink_frequency', 'PCell_Band_Indicator',
                         'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz'],
    },
    {
        'name': 'Level_3b_Uplink',
        'targets': ['PCell_Uplink_Num_RBs', 'PCell_Uplink_TB_Size',
                     'PCell_Uplink_Tx_Power_(dBm)'],
        'extra_inputs': ['Latitude', 'Longitude', 'speed_kmh', 'sin_COG', 'cos_COG',
                         'Altitude', 'Traffic Jam Factor', 'Traffic Distance',
                         'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max',
                         'PCell_SNR_1', 'PCell_SNR_2', 'PCell_freq_MHz',
                         'PCell_Downlink_frequency', 'PCell_Band_Indicator',
                         'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz'],
    },
    {
        'name': 'Level_4_QoS',
        'targets': ['datarate', 'jitter', 'Pos in Ref Round', 'target_datarate'],
        'extra_inputs': ['Latitude', 'Longitude', 'speed_kmh', 'sin_COG', 'cos_COG',
                         'Altitude', 'Traffic Jam Factor', 'Traffic Distance',
                         'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max',
                         'PCell_SNR_1', 'PCell_SNR_2', 'PCell_freq_MHz',
                         'PCell_Downlink_frequency', 'PCell_Band_Indicator',
                         'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz',
                         'PCell_Downlink_Average_MCS', 'PCell_Downlink_Num_RBs',
                         'PCell_Downlink_TB_Size',
                         'PCell_DL_RBs_MCS_Low', 'PCell_DL_RBs_MCS_Mid', 'PCell_DL_RBs_MCS_High',
                         'PCell_Uplink_Num_RBs', 'PCell_Uplink_TB_Size',
                         'PCell_Uplink_Tx_Power_(dBm)'],
    },
    {
        'name': 'Level_5_Ping',
        'targets': ['ping_ms'],
        'extra_inputs': ['Latitude', 'Longitude', 'speed_kmh', 'sin_COG', 'cos_COG',
                         'Altitude', 'Traffic Jam Factor', 'Traffic Distance',
                         'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max',
                         'PCell_SNR_1', 'PCell_SNR_2', 'PCell_freq_MHz',
                         'PCell_Downlink_frequency', 'PCell_Band_Indicator',
                         'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz',
                         'PCell_Downlink_Average_MCS', 'PCell_Downlink_Num_RBs',
                         'PCell_Downlink_TB_Size',
                         'PCell_DL_RBs_MCS_Low', 'PCell_DL_RBs_MCS_Mid', 'PCell_DL_RBs_MCS_High',
                         'PCell_Uplink_Num_RBs', 'PCell_Uplink_TB_Size',
                         'PCell_Uplink_Tx_Power_(dBm)',
                         'datarate', 'jitter', 'Pos in Ref Round', 'target_datarate'],
    },
]


In [21]:
# Post-prediction snapping rules for categorical outputs
SNAP_RULES = {
    'PCell_Downlink_frequency': [125.0, 475.0, 1300.0, 1801.0, 2850.0, 3050.0, 3749.0, 9460.0],
    'PCell_freq_MHz': [700.0, 900.0, 1800.0, 2000.0, 2100.0, 2600.0],
    'PCell_Band_Indicator': [1.0, 3.0, 7.0, 8.0, 28.0],
    'PCell_Downlink_bandwidth_MHz': [5.0, 10.0, 15.0, 20.0],
    'PCell_Uplink_bandwidth_MHz': [5.0, 10.0, 15.0, 20.0],
    'PCell_Downlink_Average_MCS': list(range(0, 30)),  # integers 0-29
}

def snap_to_nearest(values, valid_set):
    valid_arr = np.array(valid_set)
    result = np.empty_like(values)
    for i, v in enumerate(values):
        result[i] = valid_arr[np.argmin(np.abs(valid_arr - v))]
    return result


In [22]:
XGB_PARAMS = {
    'n_estimators': 500,
    'max_depth': 8,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 5,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'tree_method': 'hist',
    'random_state': 42,
    'n_jobs': -1,
}

models = {}
results = {}

# For cascaded generation: store predicted values on val/test
val_generated = df_val[BASE_INPUTS].copy()
test_generated = df_test[BASE_INPUTS].copy()

In [23]:
print("\n" + "="*80)
print("TRAINING CAUSAL CHAIN")
print("="*80)

total_start = time.time()

for level in CAUSAL_CHAIN:
    level_name = level['name']
    targets = level['targets']
    extra_inputs = level['extra_inputs']

    print(f"\n{'─'*60}")
    print(f"  {level_name}")
    print(f"  Targets: {targets}")
    print(f"  Input features: {len(BASE_INPUTS) + len(extra_inputs)}")
    print(f"{'─'*60}")

    # Build input feature list for this level
    input_cols = BASE_INPUTS + extra_inputs

    # Training uses GROUND TRUTH inputs (teacher forcing)
    X_train = df_train[input_cols]
    X_val_gt = df_val[input_cols]  # ground truth for early stopping

    # For cascaded generation on val/test, use PREDICTED inputs from previous levels
    X_val_cascade = val_generated[input_cols]
    X_test_cascade = test_generated[input_cols]

    level_models = {}

    for target in targets:
        t_start = time.time()
        y_train = df_train[target]
        y_val = df_val[target]
        y_test = df_test[target]

        model = xgb.XGBRegressor(**XGB_PARAMS, early_stopping_rounds=30)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val_gt, y_val)],
            verbose=False
        )

        # --- Evaluate: Teacher-forced (ground truth inputs) ---
        pred_val_tf = model.predict(X_val_gt)
        pred_test_tf = model.predict(df_test[input_cols])

        # --- Evaluate: Cascaded (predicted inputs from prior levels) ---
        pred_val_casc = model.predict(X_val_cascade)
        pred_test_casc = model.predict(X_test_cascade)

        # Snap categorical outputs
        if target in SNAP_RULES:
            pred_val_casc = snap_to_nearest(pred_val_casc, SNAP_RULES[target])
            pred_test_casc = snap_to_nearest(pred_test_casc, SNAP_RULES[target])
            pred_val_tf = snap_to_nearest(pred_val_tf, SNAP_RULES[target])
            pred_test_tf = snap_to_nearest(pred_test_tf, SNAP_RULES[target])

        # Store predictions for cascade
        val_generated[target] = pred_val_casc
        test_generated[target] = pred_test_casc

        # Metrics
        rmse_val_tf = np.sqrt(mean_squared_error(y_val, pred_val_tf))
        mae_val_tf = mean_absolute_error(y_val, pred_val_tf)
        rmse_test_casc = np.sqrt(mean_squared_error(y_test, pred_test_casc))
        mae_test_casc = mean_absolute_error(y_test, pred_test_casc)

        elapsed = time.time() - t_start

        results[target] = {
            'level': level_name,
            'rmse_val_tf': rmse_val_tf,
            'mae_val_tf': mae_val_tf,
            'rmse_test_cascaded': rmse_test_casc,
            'mae_test_cascaded': mae_test_casc,
            'best_iteration': model.best_iteration,
            'train_time_s': elapsed,
        }

        print(f"  {target:40s} | Val RMSE(TF): {rmse_val_tf:8.4f} | "
              f"Test RMSE(Casc): {rmse_test_casc:8.4f} | "
              f"iters: {model.best_iteration:3d} | {elapsed:.1f}s")

        level_models[target] = model

    models[level_name] = level_models

total_time = time.time() - total_start
print(f"\n{'='*80}")
print(f"TOTAL TRAINING TIME: {total_time:.1f}s ({total_time/60:.1f} min)")
print(f"{'='*80}")



TRAINING CAUSAL CHAIN

────────────────────────────────────────────────────────────
  Level_0a_GPS
  Targets: ['Latitude', 'Longitude']
  Input features: 10
────────────────────────────────────────────────────────────
  Latitude                                 | Val RMSE(TF):   0.0078 | Test RMSE(Casc):   0.0083 | iters:   0 | 1.7s
  Longitude                                | Val RMSE(TF):   0.0282 | Test RMSE(Casc):   0.0282 | iters:   4 | 0.6s

────────────────────────────────────────────────────────────
  Level_0b_Mobility
  Targets: ['speed_kmh', 'sin_COG', 'cos_COG', 'Altitude']
  Input features: 12
────────────────────────────────────────────────────────────
  speed_kmh                                | Val RMSE(TF):   1.3652 | Test RMSE(Casc):   1.6290 | iters:  34 | 1.3s
  sin_COG                                  | Val RMSE(TF):   0.3047 | Test RMSE(Casc):   1.0971 | iters:  86 | 2.0s
  cos_COG                                  | Val RMSE(TF):   0.2671 | Test RMSE(Casc):   0.569

In [24]:
if 'sin_COG' in test_generated.columns and 'cos_COG' in test_generated.columns:
    test_generated['COG_recovered'] = np.degrees(
        np.arctan2(test_generated['sin_COG'], test_generated['cos_COG'])
    ) % 360

    cog_rmse = np.sqrt(mean_squared_error(df_test['COG'], test_generated['COG_recovered']))
    # Circular MAE
    diff = np.abs(df_test['COG'].values - test_generated['COG_recovered'].values)
    circular_diff = np.minimum(diff, 360 - diff)
    cog_mae_circular = np.mean(circular_diff)

    print(f"\nCOG Recovery — Test RMSE: {cog_rmse:.4f} | Circular MAE: {cog_mae_circular:.4f}°")


COG Recovery — Test RMSE: 124.6714 | Circular MAE: 88.3006°


In [25]:
print("\n" + "="*80)
print("FULL RESULTS SUMMARY")
print("="*80)
print(f"{'Feature':45s} | {'Level':20s} | {'Val RMSE(TF)':>12s} | {'Test RMSE(Casc)':>15s} | {'Test MAE(Casc)':>14s}")
print("─" * 115)
for feat, r in sorted(results.items(), key=lambda x: x[1]['level']):
    print(f"{feat:45s} | {r['level']:20s} | {r['rmse_val_tf']:12.4f} | {r['rmse_test_cascaded']:15.4f} | {r['mae_test_cascaded']:14.4f}")



FULL RESULTS SUMMARY
Feature                                       | Level                | Val RMSE(TF) | Test RMSE(Casc) | Test MAE(Casc)
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Latitude                                      | Level_0a_GPS         |       0.0078 |          0.0083 |         0.0072
Longitude                                     | Level_0a_GPS         |       0.0282 |          0.0282 |         0.0244
speed_kmh                                     | Level_0b_Mobility    |       1.3652 |          1.6290 |         1.5445
sin_COG                                       | Level_0b_Mobility    |       0.3047 |          1.0971 |         0.8544
cos_COG                                       | Level_0b_Mobility    |       0.2671 |          0.5698 |         0.4506
Altitude                                      | Level_0b_Mobility    |       6.1014 |          6.6633 |         5.1373
precipIntensity              

In [26]:

pickle.dump(models, open('xgb_causal_chain_models.pkl', 'wb'))
pickle.dump(results, open('xgb_causal_chain_results.pkl', 'wb'))
test_generated.to_pickle('research\generation_output/xgb_test_generated.pkl')
val_generated.to_pickle('research\generation_output/xgb_val_generated.pkl')

print("\nSaved: xgb_causal_chain_models.pkl, xgb_causal_chain_results.pkl")
print("Saved: xgb_test_generated.pkl, xgb_val_generated.pkl")
print("\nDone!")


Saved: xgb_causal_chain_models.pkl, xgb_causal_chain_results.pkl
Saved: xgb_test_generated.pkl, xgb_val_generated.pkl

Done!
